In [1]:
!pip install --upgrade scipy scikit-learn

In [3]:
import pandas as pd
import numpy as np
import sklearn
import scipy

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

print("scikit-learn:", sklearn.__version__)
print("SciPy:", scipy.__version__)

# 1. Load dataset
url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)
print("\nDataset shape:", df.shape)
print(df.head())

# 2. Prepare missing values
categorical = df.select_dtypes(include="object").columns
numerical = df.select_dtypes(exclude="object").columns

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

# Target
target = "converted"

# 3. Split data
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train[target].astype(int).values
y_val = df_val[target].astype(int).values

print("\nTrain:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

# 4. Convert data to sparse matrices
def prepare_features(train_df, val_df):
    X_train_dict = train_df.drop(
        columns=[target]
    ).to_dict(orient="records")

    X_val_dict = val_df.drop(
        columns=[target]
    ).to_dict(orient="records")

    dv = DictVectorizer(sparse=True)

    X_train = dv.fit_transform(X_train_dict)
    X_val = dv.transform(X_val_dict)

    # Fix sparse index dtype for compatibility
    for X in (X_train, X_val):
        X.indices = X.indices.astype(np.int32)
        X.indptr = X.indptr.astype(np.int32)

    return X_train, X_val, dv

X_train, X_val, dv = prepare_features(df_train, df_val)

# 5. Question 1 — AUC for numerical features
print("\n========== QUESTION 1 ==========")

features = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income",
]

for feature in features:
    values = df_train[feature].astype(float).values
    auc = roc_auc_score(y_train, values)

    # Reverse direction if needed
    if auc < 0.5:
        auc = roc_auc_score(y_train, -values)

    print(f"{feature}: {auc:.3f}")

# 6. Question 2 — Logistic regression AUC
print("\n========== QUESTION 2 ==========")

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
)

model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, y_pred)
print("Validation AUC:", round(auc, 3))

# 7. Question 3 — Precision and recall
print("\n========== QUESTION 3 ==========")

thresholds = np.arange(0.0, 1.001, 0.01)
pr_results = []

for threshold in thresholds:
    pred = (y_pred >= threshold).astype(int)

    precision = precision_score(
        y_val, pred, zero_division=0
    )
    recall = recall_score(
        y_val, pred, zero_division=0
    )

    pr_results.append(
        (threshold, precision, recall)
    )

# Find the threshold where precision and recall are closest
q3 = min(
    pr_results,
    key=lambda x: abs(x[1] - x[2])
)

print("Threshold:", round(q3[0], 2))
print("Precision:", round(q3[1], 3))
print("Recall:", round(q3[2], 3))

# 8. Question 4 — F1 score
print("\n========== QUESTION 4 ==========")

f1_results = []

for threshold in thresholds:
    pred = (y_pred >= threshold).astype(int)

    f1 = f1_score(
        y_val, pred, zero_division=0
    )

    f1_results.append((threshold, f1))

q4 = max(f1_results, key=lambda x: x[1])

print("Best threshold:", round(q4[0], 2))
print("Maximum F1:", round(q4[1], 3))

# 9. Question 5 — Five-fold cross-validation
print("\n========== QUESTION 5 ==========")

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=1,
)

cv_scores = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(df_full_train), start=1
):
    fold_train = df_full_train.iloc[train_idx]
    fold_val = df_full_train.iloc[val_idx]

    X_tr, X_va, _ = prepare_features(
        fold_train, fold_val
    )

    y_tr = fold_train[target].astype(int).values
    y_va = fold_val[target].astype(int).values

    m = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
    )

    m.fit(X_tr, y_tr)
    probabilities = m.predict_proba(X_va)[:, 1]

    score = roc_auc_score(y_va, probabilities)
    cv_scores.append(score)

    print(f"Fold {fold} AUC: {score:.3f}")

print("Mean AUC:", round(np.mean(cv_scores), 3))
print("Standard deviation:", round(np.std(cv_scores), 3))

# 10. Question 6 — Tune regularization parameter C
print("\n========== QUESTION 6 ==========")

for C in [0.000001, 0.001, 1.0]:
    scores = []

    for train_idx, val_idx in kf.split(df_full_train):
        fold_train = df_full_train.iloc[train_idx]
        fold_val = df_full_train.iloc[val_idx]

        X_tr, X_va, _ = prepare_features(
            fold_train, fold_val
        )

        y_tr = fold_train[target].astype(int).values
        y_va = fold_val[target].astype(int).values

        m = LogisticRegression(
            solver="liblinear",
            C=C,
            max_iter=1000,
        )

        m.fit(X_tr, y_tr)
        probabilities = m.predict_proba(X_va)[:, 1]

        scores.append(
            roc_auc_score(y_va, probabilities)
        )

    print(
        f"C={C}: mean AUC={np.mean(scores):.3f}, "
        f"std={np.std(scores):.3f}"
    )

print("\nAll calculations completed.")


scikit-learn: 1.9.1
SciPy: 1.18.1

Dataset shape: (5000, 9)
      lead_source       industry employment_status       location  \
0  organic_search     technology          employed         europe   
1    social_media     technology          employed  south_america   
2        referral         retail          employed         europe   
3        paid_ads  manufacturing           student            NaN   
4        referral  manufacturing           student  north_america   

   annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0        88160.0                         3                  4        0.64   
1        72688.0                         3                  7        0.72   
2        44697.0                         3                  4        0.58   
3            NaN                         3                  6        0.57   
4        24062.0                         4                  5        0.62   

   converted  
0          1  
1          1  
2          1  
3 

/tmp/ipykernel_14519/1968877825.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df.select_dtypes(include="object").columns



========== QUESTION 1 ==========
lead_score: 0.788
number_of_courses_viewed: 0.723
interaction_count: 0.765
annual_income: 0.608

========== QUESTION 2 ==========
Validation AUC: 0.732

========== QUESTION 3 ==========
Threshold: 0.97
Precision: 0.0
Recall: 0.0

========== QUESTION 4 ==========
Best threshold: 0.41
Maximum F1: 0.758

========== QUESTION 5 ==========
Fold 1 AUC: 0.731
Fold 2 AUC: 0.730
Fold 3 AUC: 0.736
Fold 4 AUC: 0.741
Fold 5 AUC: 0.721
Mean AUC: 0.732
Standard deviation: 0.007

========== QUESTION 6 ==========
C=1e-06: mean AUC=0.617, std=0.019
C=0.001: mean AUC=0.749, std=0.010
C=1.0: mean AUC=0.732, std=0.007

All calculations completed.
